# Phase 1 — EIA capacity factor targets

**Phase 1 screening:** Keep **plant × year** rows; snap filter ≤ 5 km; stricter CF outlier cap.

| Input | Output |
|-------|--------|
| `data/model_data/NY_Greation_Data_with_nlcd.csv` | `data/processed/eia_facility_annual.parquet` |
| `data/processed/weather_cell_features.parquet` | `data/processed/eia_facility_features.parquet` |

**CF formula:** `Net Generation (MWh) / (Nameplate MW × 8760)`

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import geopandas as gpd
import polars as pl
import yaml
from shapely.geometry import Point

ROOT = Path('../..').resolve()
CONFIG_PATH = ROOT / 'config' / 'paths.yaml'
EIA_PATH = ROOT / 'data' / 'model_data' / 'NY_Greation_Data_with_nlcd.csv'
WEATHER_PATH = ROOT / 'data' / 'processed' / 'weather_cell_features.parquet'
ANNUAL_OUT = ROOT / 'data' / 'processed' / 'eia_facility_annual.parquet'
FEATURES_OUT = ROOT / 'data' / 'processed' / 'eia_facility_features.parquet'

RENEWABLE_FUELS = {'SUN': 'solar', 'WND': 'wind'}
CF_OUTLIER_MAX = 1.05  # outlier cap (stricter than the prior 1.2 threshold)
SNAP_MAX_DISTANCE_M = 5000  # drop plant-years snapped farther than 5 km
TARGET_CRS = 'EPSG:32618'

with CONFIG_PATH.open(encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

ANNUAL_OUT.parent.mkdir(parents=True, exist_ok=True)

if not EIA_PATH.exists():
    raise FileNotFoundError(f'Missing EIA file: {EIA_PATH}')
if not WEATHER_PATH.exists():
    raise FileNotFoundError(f'Run 01_ingest_weather.ipynb first: {WEATHER_PATH}')

print('EIA input :', EIA_PATH)
print('Weather   :', WEATHER_PATH)

EIA input : /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/model_data/NY_Greation_Data_with_nlcd.csv
Weather   : /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/weather_cell_features.parquet


In [2]:
eia = pl.read_csv(EIA_PATH, infer_schema_length=10_000)

renewable = eia.filter(pl.col('Reported Fuel Type Code').is_in(list(RENEWABLE_FUELS.keys())))

annual = (
    renewable
    .with_columns([
        pl.col('Reported Fuel Type Code').alias('fuel_code'),
        pl.col('Plant Nameplate Capacity (MW)').alias('nameplate_mw'),
        pl.col('Net Generation (Megawatthours)').alias('net_generation_mwh'),
        (pl.col('Net Generation (Megawatthours)') / (pl.col('Plant Nameplate Capacity (MW)') * 8760.0)).alias('capacity_factor'),
        pl.col('Reported Fuel Type Code').replace(RENEWABLE_FUELS).alias('technology'),
    ])
    .filter(pl.col('nameplate_mw') > 0)
    .filter(pl.col('Latitude').is_not_null() & pl.col('Longitude').is_not_null())
    .select([
        pl.col('Plant Id').alias('plant_id'),
        pl.col('Plant Name').alias('plant_name'),
        pl.col('YEAR').alias('year'),
        'technology', 'fuel_code', pl.col('Reported Prime Mover').alias('prime_mover'),
        'Latitude', 'Longitude', 'County', 'State',
        'nameplate_mw', 'net_generation_mwh', 'capacity_factor',
        'nlcd_code', 'nlcd_class',
    ])
    .with_columns([
        (pl.col('capacity_factor') > CF_OUTLIER_MAX).alias('cf_outlier_flag'),
        pl.col('capacity_factor').is_between(0, CF_OUTLIER_MAX).alias('cf_valid_flag'),
    ])
)

annual.write_parquet(ANNUAL_OUT)
print(f'Wrote {len(annual):,} facility-year rows -> {ANNUAL_OUT}')
print(annual.group_by('technology').agg(pl.len().alias('rows'), pl.col('plant_id').n_unique().alias('plants')))

Wrote 2,794 facility-year rows -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/eia_facility_annual.parquet
shape: (2, 3)
┌────────────┬──────┬────────┐
│ technology ┆ rows ┆ plants │
│ ---        ┆ ---  ┆ ---    │
│ str        ┆ u32  ┆ u32    │
╞════════════╪══════╪════════╡
│ solar      ┆ 2424 ┆ 611    │
│ wind       ┆ 370  ┆ 35     │
└────────────┴──────┴────────┘


In [3]:
# CF QA
annual.select([
    pl.len().alias('n_rows'),
    pl.col('capacity_factor').mean().alias('cf_mean'),
    pl.col('capacity_factor').median().alias('cf_median'),
    pl.col('capacity_factor').max().alias('cf_max'),
    pl.col('cf_outlier_flag').sum().alias('n_outliers_gt_1_2'),
    pl.col('cf_valid_flag').sum().alias('n_valid_cf'),
])

n_rows,cf_mean,cf_median,cf_max,n_outliers_gt_1_2,n_valid_cf
u32,f64,f64,f64,u32,u32
2794,0.030367,0.024113,0.470911,0,2793


In [4]:
# Snap each facility to nearest 10 km weather grid cell
weather = pl.read_parquet(WEATHER_PATH)

plants_pdf = (
    annual
    .select(['plant_id', 'Latitude', 'Longitude'])
    .unique(subset=['plant_id'])
    .rename({'Latitude': 'latitude', 'Longitude': 'longitude'})
    .to_pandas()
)

weather_pdf = (
    weather.select(['latitude', 'longitude'])
    .rename({'latitude': 'grid_latitude', 'longitude': 'grid_longitude'})
    .to_pandas()
)

plants_gdf = gpd.GeoDataFrame(
    plants_pdf,
    geometry=[Point(r.longitude, r.latitude) for r in plants_pdf.itertuples()],
    crs='EPSG:4326',
).to_crs(TARGET_CRS)

weather_gdf = gpd.GeoDataFrame(
    weather_pdf,
    geometry=[Point(r.grid_longitude, r.grid_latitude) for r in weather_pdf.itertuples()],
    crs='EPSG:4326',
).to_crs(TARGET_CRS)

nearest = gpd.sjoin_nearest(
    plants_gdf,
    weather_gdf[['grid_latitude', 'grid_longitude', 'geometry']],
    how='left',
    distance_col='dist_to_cell_m',
)

plant_cell_map = pl.from_pandas(
    nearest.drop(columns=['geometry', 'index_right'], errors='ignore')[
        ['plant_id', 'grid_latitude', 'grid_longitude', 'dist_to_cell_m']
    ]
)

features = (
    annual
    .join(plant_cell_map, on='plant_id', how='left')
    .join(
        weather.rename({'latitude': 'grid_latitude', 'longitude': 'grid_longitude'}),
        on=['grid_latitude', 'grid_longitude'],
        how='left',
        suffix='_wx',
    )
    .with_columns(
        (pl.col('dist_to_cell_m') <= SNAP_MAX_DISTANCE_M).alias('snap_within_5km'),
    )
    .filter(pl.col('snap_within_5km'))
)

features.write_parquet(FEATURES_OUT)
print(f'Wrote {len(features):,} rows x {len(features.columns)} cols -> {FEATURES_OUT}')
print('Snap filter: <=', SNAP_MAX_DISTANCE_M, 'm')
print('Mean snap distance (m):', round(features['dist_to_cell_m'].mean(), 1))
print('Dropped plant-years beyond snap limit:', annual.height - features.height)

Wrote 2,499 rows x 103 cols -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/eia_facility_features.parquet
Snap filter: <= 5000 m
Mean snap distance (m): 3113.0
Dropped plant-years beyond snap limit: 295


In [5]:
# Preview — solar vs wind CF by year
features.filter(pl.col('cf_valid_flag')).group_by(['technology', 'year']).agg(
    pl.col('capacity_factor').mean().alias('mean_cf'),
    pl.len().alias('n'),
).sort(['technology', 'year']).tail(10)

technology,year,mean_cf,n
str,i64,f64,u32
"""wind""",2016,0.020154,20
"""wind""",2017,0.021903,21
"""wind""",2018,0.018873,23
"""wind""",2019,0.02132,24
"""wind""",2020,0.022965,24
"""wind""",2021,0.021022,26
"""wind""",2022,0.025058,26
"""wind""",2023,0.031044,31
"""wind""",2024,0.044326,30
